# Toxicity v2 — 22 ADMET properties (TDC benchmark)

**ADMET = dawa body me kya karegi:**
- **A**bsorption: pet se khoon me jayegi? (HIA, Caco-2, Bioavailability, Pgp)
- **D**istribution: dimaag tak pahunchegi? protein se kitni chipkegi? (BBB, PPBR, VDss)
- **M**etabolism: liver ke CYP enzymes ise todenge/rokenge? (CYP2C9, 2D6, 3A4)
- **E**xcretion: body se kitni der me niklegi? (Half-life, Clearance)
- **T**oxicity: dil pe asar (hERG), DNA damage (AMES), liver damage (DILI), zehar ki matra (LD50)
- Plus: Solubility, Lipophilicity

**Ye "TDC ADMET Benchmark Group" hai** — duniya bhar ke research models isi pe compare hote hain (public leaderboard).
Notebook official tareeke se (5 seeds, official test split) score nikalega aur **top leaderboard model** se compare karega.

**Chalane se pehle:** Runtime → Change runtime type → **T4 GPU** (molecule AI ke liye; na mile to CPU pe bhi chalega, bas dheema) → **Run all** → Drive "Allow".
Time: ~30–60 minute.

## 1. Tools install

In [ ]:
import subprocess, sys
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print(("✅ " if r.returncode == 0 else "❌ ") + " ".join(args))
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode == 0

pip("rdkit")
pip("transformers")
# PyTDC bina dependencies ke — taaki Colab ke numpy/pandas na badlein
pip("--no-deps", "PyTDC")
pip("fuzzywuzzy", "huggingface_hub", "tqdm", "seaborn", "requests", "scikit-learn")

In [ ]:
import os, time, gzip, shutil, warnings
warnings.filterwarnings("ignore")
try:
    import numpy as np, pandas as pd, torch, xgboost as xgb
    from rdkit import Chem, RDLogger
    from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
    from tdc.benchmark_group import admet_group
except Exception as e:
    import traceback; traceback.print_exc()
    raise SystemExit(f"Import error: {e}\n→ Section 1 ke output me koi ❌ hai? Uska screenshot Claude ko bhejo. "
                     "Agar sab ✅ hai to: Runtime → Restart session, phir Run all.")
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score, mean_absolute_error
import matplotlib.pyplot as plt
RDLogger.DisableLog("rdApp.*")
GPU = torch.cuda.is_available(); DEVICE = "cuda" if GPU else "cpu"
print("Sab tools ready ✅ |", "GPU mila ✅" if GPU else "GPU nahi — CPU pe chalega (dheema)")

## 2. Google Drive (results aur models save karne ke liye)

In [ ]:
SAVE = "admet_v2"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/ai_drug_platform/admet_v2"
except Exception as e:
    print("Drive nahi juda, local folder:", e)
os.makedirs(os.path.join(SAVE, "models"), exist_ok=True)
print("Save folder:", SAVE)

## 3. TDC ADMET data (22 datasets)

In [ ]:
group = admet_group(path="tdc_data/")
NAMES = list(group.dataset_names)
print(len(NAMES), "datasets:", NAMES)

# official metric har dataset ka (TDC leaderboard wala)
METRIC = {
    "caco2_wang": "MAE", "lipophilicity_astrazeneca": "MAE", "solubility_aqsoldb": "MAE",
    "ppbr_az": "MAE", "ld50_zhu": "MAE",
    "vdss_lombardo": "Spearman", "half_life_obach": "Spearman",
    "clearance_hepatocyte_az": "Spearman", "clearance_microsome_az": "Spearman",
    "hia_hou": "AUROC", "pgp_broccatelli": "AUROC", "bioavailability_ma": "AUROC", "bbb_martins": "AUROC",
    "herg": "AUROC", "ames": "AUROC", "dili": "AUROC", "cyp3a4_substrate_carbonmangels": "AUROC",
    "cyp2c9_veith": "AUPRC", "cyp2d6_veith": "AUPRC", "cyp3a4_veith": "AUPRC",
    "cyp2c9_substrate_carbonmangels": "AUPRC", "cyp2d6_substrate_carbonmangels": "AUPRC",
}
# leaderboard ke ek top model (MapLight) ke approximate scores — sirf comparison ke liye
REF = {
    "caco2_wang": 0.276, "lipophilicity_astrazeneca": 0.539, "solubility_aqsoldb": 0.792, "ppbr_az": 7.66,
    "ld50_zhu": 0.621, "vdss_lombardo": 0.707, "half_life_obach": 0.562, "clearance_hepatocyte_az": 0.466,
    "clearance_microsome_az": 0.626, "hia_hou": 0.986, "pgp_broccatelli": 0.930, "bbb_martins": 0.916,
    "herg": 0.871, "ames": 0.868, "dili": 0.887, "cyp3a4_substrate_carbonmangels": 0.650,
    "cyp2c9_veith": 0.783, "cyp2d6_veith": 0.723, "cyp3a4_veith": 0.881,
    "cyp2c9_substrate_carbonmangels": 0.415, "cyp2d6_substrate_carbonmangels": 0.713,
}
LOG_TARGET = {"vdss_lombardo", "half_life_obach", "clearance_hepatocyte_az", "clearance_microsome_az"}
missing = [n for n in NAMES if n not in METRIC]
assert not missing, f"Naye datasets jinka metric pata nahi: {missing} — screenshot Claude ko bhejo"

def split(name):
    b = group.get(name)
    if "train_val" in b:
        tv = b["train_val"]
    else:
        tv = pd.concat([b["train"], b["valid"]])
    return tv.reset_index(drop=True), b["test"].reset_index(drop=True)

DATA = {n: split(n) for n in NAMES}
for n, (tv, te) in DATA.items():
    print(f"{n:34s} train {len(tv):6,}  test {len(te):5,}  metric {METRIC[n]}")

## 4. Molecule features
- Morgan **count** fingerprint (2048) — kaun se tukde kitni baar
- MACCS keys (167) — standard chemistry patterns
- ~210 RDKit descriptors — weight, charge, logP, rings...
- **ChemBERTa** (384) — molecule AI embedding

Saare unique molecules ek baar featurize honge aur Drive me cache ho jayenge.

In [ ]:
from transformers import AutoTokenizer, AutoModel
CHEM_NAME = "DeepChem/ChemBERTa-77M-MTR"
mfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def rdkit_feats(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None
    fp = mfp.GetCountFingerprintAsNumPy(m).astype(np.float32)
    mac = np.array(list(MACCSkeys.GenMACCSKeys(m)), dtype=np.float32)
    d = []
    for _, fn in Descriptors._descList:
        try: d.append(float(fn(m)))
        except Exception: d.append(np.nan)
    d = np.nan_to_num(np.clip(np.array(d, np.float64), -1e6, 1e6)).astype(np.float32)
    return np.concatenate([fp, mac, d])

@torch.no_grad()
def chemberta(smiles, batch=128):
    tok = AutoTokenizer.from_pretrained(CHEM_NAME)
    model = AutoModel.from_pretrained(CHEM_NAME).to(DEVICE).eval()
    out = []
    for i in range(0, len(smiles), batch):
        t = tok(smiles[i:i + batch], return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
        h = model(**t).last_hidden_state
        msk = t["attention_mask"].unsqueeze(-1).float()
        out.append(((h * msk).sum(1) / msk.sum(1)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

all_smiles = sorted({s for tv, te in DATA.values() for s in pd.concat([tv, te])["Drug"].astype(str)})
cache = os.path.join(SAVE, "features_cache.npz")
if os.path.exists(cache):
    z = np.load(cache, allow_pickle=True)
    if list(z["smiles"]) == all_smiles:
        FEAT = dict(zip(all_smiles, z["X"])); print("Features Drive se load ✅", len(FEAT))
    else:
        FEAT = None
else:
    FEAT = None

if FEAT is None:
    t0 = time.time()
    rd = [rdkit_feats(s) for s in all_smiles]
    ok = [k for k, v in enumerate(rd) if v is not None]
    width = len(rd[ok[0]])
    rd = np.stack([v if v is not None else np.zeros(width, np.float32) for v in rd])
    print(f"RDKit features done ({time.time()-t0:.0f}s), ab ChemBERTa...")
    cb = chemberta(all_smiles)
    X_all = np.hstack([rd, cb]).astype(np.float32)
    np.savez_compressed(cache, smiles=np.array(all_smiles, dtype=object), X=X_all)
    FEAT = dict(zip(all_smiles, X_all))
    print(f"Features bane ✅ {X_all.shape} ({time.time()-t0:.0f}s)")

def X_of(df):
    return np.stack([FEAT[str(s)] for s in df["Drug"]])

## 5. Training + official evaluation (22 datasets × 5 seeds)
Har dataset pe 5 baar alag seed se train hota hai, phir **mean ± std** — bilkul leaderboard ke rule jaisa.

In [ ]:
def make(kind, seed):
    common = dict(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                  colsample_bytree=0.3, min_child_weight=1, tree_method="hist",
                  device=DEVICE, random_state=seed, n_jobs=-1)
    return xgb.XGBRegressor(**common) if kind == "reg" else xgb.XGBClassifier(eval_metric="logloss", **common)

def score(metric, y, p):
    if metric == "MAE": return mean_absolute_error(y, p)
    if metric == "Spearman": return spearmanr(y, p).correlation
    if metric == "AUROC": return roc_auc_score(y, p)
    return average_precision_score(y, p)

# TDC ke official evaluator ke liye har seed ki predictions sambhal kar rakhni
# hoti hain: list of dicts, har dict me saare datasets.
# Pehle humne metrics KHUD sklearn se nikale the - wo leaderboard submission ke
# liye nahi chalta, guideline saaf kehti hai ki group.evaluate_many use karo.
# Dono rakhte hain, taaki farak dikh jaye.
SEEDS = [1, 2, 3, 4, 5]
PREDS = {s: {} for s in SEEDS}
rows, t0 = [], time.time()
for n in NAMES:
    tv, te = DATA[n]
    Xtr, Xte = X_of(tv), X_of(te)
    ytr, yte = tv["Y"].values.astype(float), te["Y"].values.astype(float)
    kind = "clf" if METRIC[n] in ("AUROC", "AUPRC") else "reg"
    scores = []
    for seed in [1, 2, 3, 4, 5]:
        m = make(kind, seed)
        if kind == "clf":
            m.fit(Xtr, ytr.astype(int)); p = m.predict_proba(Xte)[:, 1]
        elif n in LOG_TARGET:
            m.fit(Xtr, np.log1p(np.clip(ytr, 0, None))); p = np.expm1(m.predict(Xte))
        else:
            m.fit(Xtr, ytr); p = m.predict(Xte)
        scores.append(score(METRIC[n], yte, p))
        PREDS[seed][n] = p
    mean, std = float(np.mean(scores)), float(np.std(scores))
    ref = REF.get(n)
    better_low = METRIC[n] == "MAE"
    if ref is None: verdict = "—"
    elif (mean <= ref) if better_low else (mean >= ref): verdict = "🏆 barabar/behtar"
    elif abs(mean - ref) / abs(ref) <= 0.05: verdict = "✅ 5% ke andar"
    else: verdict = "⬜ peeche"
    rows.append({"dataset": n, "metric": METRIC[n], "humara": round(mean, 3), "± std": round(std, 3),
                 "top model (approx)": ref, "verdict": verdict})
    print(f"{n:34s} {METRIC[n]:8s} {mean:.3f} ± {std:.3f}   ref {ref}   {verdict}   ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(SAVE, "admet_v2_results.csv"), index=False)
print("\nSummary:", res["verdict"].value_counts().to_dict())

## 6. Official TDC evaluation — **submission ke liye yahi number chahiye**

Upar ke numbers humne khud sklearn se nikale. Leaderboard ki guideline saaf
kehti hai: **TDC ka apna evaluator** use karo (`group.evaluate_many`), aur
**TDC ka test set**. Isliye wahi chalate hain, aur dono numbers side-by-side
rakhte hain — agar farak aaya to wo khud ek kaam ki baat hai.

**Iska screenshot Claude ko bhejna hai.**

In [ ]:
predictions_list = [PREDS[s] for s in SEEDS]
missing = [n for n in NAMES if any(n not in d for d in predictions_list)]
assert not missing, f"in datasets ki predictions adhoori hain: {missing}"
print(f"{len(predictions_list)} seeds x {len(NAMES)} datasets bhej rahe hain...\n")

official = group.evaluate_many(predictions_list)
print("=== TDC ka official output ===")
print(official)

# official ka roop {dataset: [mean, std]} hota hai; defensive tareeke se padho
off_rows = []
for n in NAMES:
    key = n if n in official else next((k for k in official if k.lower() == n.lower()), None)
    if key is None:
        off_rows.append({"dataset": n, "official_mean": None, "official_std": None})
        continue
    v = official[key]
    if isinstance(v, dict): v = list(v.values())[0]
    if isinstance(v, (list, tuple)):
        off_rows.append({"dataset": n, "official_mean": float(v[0]),
                         "official_std": float(v[1]) if len(v) > 1 else 0.0})
    else:
        off_rows.append({"dataset": n, "official_mean": float(v), "official_std": 0.0})

off = pd.DataFrame(off_rows)
cmp_tbl = res.merge(off, on="dataset")
cmp_tbl["farak"] = (cmp_tbl.official_mean - cmp_tbl.humara).round(4)
print("\n=== apna hisaab vs TDC ka official ===")
print(cmp_tbl[["dataset", "metric", "humara", "official_mean", "official_std",
               "farak", "top model (approx)"]].to_string(index=False))

big = cmp_tbl[cmp_tbl.farak.abs() > 0.01]
if len(big):
    print(f"\n⚠️ {len(big)} datasets me 0.01 se zyada farak - inko dekhna hoga:")
    print(big[["dataset", "metric", "humara", "official_mean", "farak"]].to_string(index=False))
else:
    print("\n✅ Sab datasets 0.01 ke andar - apna hisaab official se mel khata hai")

cmp_tbl.to_csv(os.path.join(SAVE, "admet_official_vs_ours.csv"), index=False)
json.dump({"official": {r["dataset"]: [r["official_mean"], r["official_std"]]
                        for r in off_rows},
           "n_seeds": len(SEEDS)},
          open(os.path.join(SAVE, "admet_official.json"), "w"), indent=2)
print(f"\nsave: {SAVE}/admet_official.json")

**Submission ke liye:** `admet_official.json` me jo numbers hain **wahi** form me
jaane chahiye — upar ka apna hisaab nahi, chahe wo lagbhag same ho.

In [ ]:
print("Ab ye poora output Claude ko bhejo - official numbers submission me jayenge.")
res

**Kaise padhein:**
- **MAE** = galti, **kam behtar**. **AUROC / AUPRC / Spearman** = **zyada behtar**.
- 🏆 = leaderboard ke top model ke barabar ya upar · ✅ = 5% ke andar · ⬜ = peeche.
- "top model" ke numbers approximate hain (MapLight, ek public benchmark site se) — asli leaderboard: tdcommons.ai.

## 6. Graph: top model ke mukable kahan khade hain

In [ ]:
r = res.dropna(subset=["top model (approx)"]).copy()
# sabko "zyada = behtar" scale pe: MAE ke liye ref/humara, baaki ke liye humara/ref
r["ratio"] = np.where(r.metric == "MAE", r["top model (approx)"] / r["humara"], r["humara"] / r["top model (approx)"])
r = r.sort_values("ratio")
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(r.dataset, r.ratio, color=["#2f6fdf" if v >= 0.95 else "#9aa5b1" for v in r.ratio])
ax.axvline(1.0, color="black", lw=1); ax.axvline(0.95, color="gray", ls="--", lw=1)
ax.set_xlabel("Humara score ÷ top model (1.0 = barabar, zyada = behtar)")
ax.set_title("ADMET v2 vs leaderboard top model")
plt.tight_layout(); plt.show()

## 7. Final models (train + test dono pe) — app ke liye Drive me save

In [ ]:
import json
for n in NAMES:
    tv, te = DATA[n]
    full = pd.concat([tv, te]).reset_index(drop=True)
    X, y = X_of(full), full["Y"].values.astype(float)
    kind = "clf" if METRIC[n] in ("AUROC", "AUPRC") else "reg"
    m = make(kind, 42)
    if kind == "clf": m.fit(X, y.astype(int))
    elif n in LOG_TARGET: m.fit(X, np.log1p(np.clip(y, 0, None)))
    else: m.fit(X, y)
    tmp = "/tmp/_m.ubj"; m.save_model(tmp)
    with open(tmp, "rb") as fi, gzip.open(os.path.join(SAVE, "models", f"{n}.ubj.gz"), "wb") as fo:
        shutil.copyfileobj(fi, fo)
json.dump({"names": NAMES, "metric": METRIC, "log_target": sorted(LOG_TARGET), "chem_name": CHEM_NAME,
           "feature_width": int(len(next(iter(FEAT.values()))))},
          open(os.path.join(SAVE, "models", "config.json"), "w"), indent=1)
print("22 final models save ✅ ->", os.path.join(SAVE, "models"))

## Next
**Section 5 ki table** (ya print hui lines) ka screenshot/text Claude ko bhejo.
Phir ye 22 properties **Streamlit app** me jodenge → Phase 1 poora ✅